<a href="https://colab.research.google.com/github/d-monkey-CG/Dallas_County_ResidentialProperty_Stats/blob/main/Dallas_CB_HIncome.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Census Bureau Data: Median Income by Household (Zipcode and Year)**

This notebook brings together data from the Census Bureau's ACS datasets. The objective is to put together a dataset that has the median income per household from 2020 to 2024 by zipcode and year. Then we will add this data to the existing dataset of DCAD home appraisal values to complete our dataset before creating an analytics dashboard in Looker Studio.

I am hoping that including median income per household by zipcode will offer some perspective on the affordability (or lack thereof) of residential property. The name of the exact report I used is MEDIAN HOUSEHOLD INCOME IN THE PAST 12 MONTHS (IN 20XX INFLATION-ADJUSTED DOLLARS), I filtered the report by tract.

In [ ]:
import requests
import pandas as pd
import re

def load_census_json(url: str) -> pd.DataFrame:
    """
    Fetch JSON data from the Census API, convert to a DataFrame,
    and automatically extract the year from the URL.
    """
    # Extract year from the URL
    match = re.search(r'/data/(20\d{2})/', url)
    if match:
        year = int(match.group(1))
    else:
        raise ValueError("Could not extract year from the URL.")

    # Request data
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()

    # Load into DataFrame
    df = pd.DataFrame(data[1:], columns=data[0])

    # Add the year column
    df["year"] = year

    return df

In [ ]:
# To load the datasets and create the df including the year as a new column
# https://api.census.gov/data/2024/acs/acs5 -- Metadata Link

df_2020 = load_census_json("https://api.census.gov/data/2020/acs/acs5?get=group(B19013)&ucgid=pseudo(0500000US48113$1400000)")
df_2021 = load_census_json("https://api.census.gov/data/2021/acs/acs5?get=group(B19013)&ucgid=pseudo(0500000US48113$1400000)")
df_2022 = load_census_json("https://api.census.gov/data/2022/acs/acs5?get=group(B19013)&ucgid=pseudo(0500000US48113$1400000)")
df_2023 = load_census_json("https://api.census.gov/data/2023/acs/acs5?get=group(B19013)&ucgid=pseudo(0500000US48113$1400000)")
df_2024 = load_census_json("https://api.census.gov/data/2024/acs/acs5?get=group(B19013)&ucgid=pseudo(0500000US48113$1400000)")

In [ ]:
# To combine the dataframes into a single large dataframe

df_CB = pd.concat([df_2020, df_2021, df_2022, df_2023, df_2024], axis=0, ignore_index=True)

df_CB.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3225 entries, 0 to 3224
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   B19013_001E   3225 non-null   object
 1   B19013_001EA  66 non-null     object
 2   B19013_001M   3225 non-null   object
 3   B19013_001MA  66 non-null     object
 4   GEO_ID        3225 non-null   object
 5   NAME          3225 non-null   object
 6   ucgid         3225 non-null   object
 7   year          3225 non-null   int64 
dtypes: int64(1), object(7)
memory usage: 201.7+ KB


In [ ]:
df_CB.head()

,B19013_001E,B19013_001EA,B19013_001M,B19013_001MA,GEO_ID,NAME,ucgid,year
0,127625,None,35632,None,1400000US48113000100,"Census Tract 1, Dallas County, Texas",1400000US48113000100,2020
1,156000,None,32552,None,1400000US48113000201,"Census Tract 2.01, Dallas County, Texas",1400000US48113000201,2020
2,123571,None,19344,None,1400000US48113000202,"Census Tract 2.02, Dallas County, Texas",1400000US48113000202,2020
3,86741,None,7950,None,1400000US48113000300,"Census Tract 3, Dallas County, Texas",1400000US48113000300,2020
4,52347,None,5354,None,1400000US48113000401,"Census Tract 4.01, Dallas County, Texas",1400000US48113000401,2020


In [ ]:
# To change the column names on the dataframe to be more human readable, I based the names on the actual table from the Census Bureau site.

df_CB = df_CB.rename(columns={'B19013_001E':'Median_H_Inc','NAME':'TRACT'})

df_CB.head()

,Median_H_Inc,B19013_001EA,B19013_001M,B19013_001MA,GEO_ID,TRACT,ucgid,year
0,127625,None,35632,None,1400000US48113000100,"Census Tract 1, Dallas County, Texas",1400000US48113000100,2020
1,156000,None,32552,None,1400000US48113000201,"Census Tract 2.01, Dallas County, Texas",1400000US48113000201,2020
2,123571,None,19344,None,1400000US48113000202,"Census Tract 2.02, Dallas County, Texas",1400000US48113000202,2020
3,86741,None,7950,None,1400000US48113000300,"Census Tract 3, Dallas County, Texas",1400000US48113000300,2020
4,52347,None,5354,None,1400000US48113000401,"Census Tract 4.01, Dallas County, Texas",1400000US48113000401,2020


In [ ]:
# To drop the columns we don't need (i.e. margin of error, columns where values=none, GEO_ID, etc.)

df_CB = df_CB.drop(columns=['B19013_001EA','B19013_001M','B19013_001MA','ucgid'])

df_CB.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3225 entries, 0 to 3224
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Median_H_Inc  3225 non-null   object
 1   GEO_ID        3225 non-null   object
 2   TRACT         3225 non-null   object
 3   year          3225 non-null   int64 
dtypes: int64(1), object(3)
memory usage: 100.9+ KB


In [ ]:
# To re-order the columns to be more human readable

new_order = ['year','TRACT','GEO_ID','Median_H_Inc']

df_CB = df_CB[new_order]

df_CB.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3225 entries, 0 to 3224
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   year          3225 non-null   int64 
 1   TRACT         3225 non-null   object
 2   GEO_ID        3225 non-null   object
 3   Median_H_Inc  3225 non-null   object
dtypes: int64(1), object(3)
memory usage: 100.9+ KB


In [ ]:
# To drop all rows with useless values in them

df_CB = df_CB[~(df_CB['Median_H_Inc'] == '-666666666')]

# To have only the 11 digit TRACT_ID show in a separate column (this will be used to join with zipcodes later)

df_CB.loc[:, 'TRACT_ID'] = df_CB['GEO_ID'].str.extract(r'(48\d{9})', expand=False)

df_CB.head()

,year,TRACT,GEO_ID,Median_H_Inc,TRACT_ID
0,2020,"Census Tract 1, Dallas County, Texas",1400000US48113000100,127625,48113000100
1,2020,"Census Tract 2.01, Dallas County, Texas",1400000US48113000201,156000,48113000201
2,2020,"Census Tract 2.02, Dallas County, Texas",1400000US48113000202,123571,48113000202
3,2020,"Census Tract 3, Dallas County, Texas",1400000US48113000300,86741,48113000300
4,2020,"Census Tract 4.01, Dallas County, Texas",1400000US48113000401,52347,48113000401


In [ ]:
# Extract tract number and county

df_CB[['TRACT', 'county']] = df_CB['TRACT'].str.extract(
    r'Census Tract\s+([\d\.]+)[,;]\s+([\w\s]+) County'
)
df_CB.head()

,year,TRACT,GEO_ID,Median_H_Inc,TRACT_ID,county
0,2020,1,1400000US48113000100,127625,48113000100,Dallas
1,2020,2.01,1400000US48113000201,156000,48113000201,Dallas
2,2020,2.02,1400000US48113000202,123571,48113000202,Dallas
3,2020,3,1400000US48113000300,86741,48113000300,Dallas
4,2020,4.01,1400000US48113000401,52347,48113000401,Dallas


In [ ]:
df_CB.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3207 entries, 0 to 3221
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   year          3207 non-null   int64 
 1   TRACT         3207 non-null   object
 2   GEO_ID        3207 non-null   object
 3   Median_H_Inc  3207 non-null   object
 4   TRACT_ID      3207 non-null   object
 5   county        3207 non-null   object
dtypes: int64(1), object(5)
memory usage: 175.4+ KB


In [ ]:
df_CB['county'].value_counts()

,count
county,
Dallas,3207


In [ ]:
df_CB.to_csv('Dallas_County_CBHInc.csv', index=False)